# Geotechnical Data Cleaning & Validation
## Phase 2: Rigorous Cleaning with Domain-Specific Logic

**Objective:** Apply geotechnical engineering principles to ensure data quality and physical consistency.

**Key Operations:**
1. Atterberg Limits consistency validation ($PI = LL - PL$)
2. Physical boundary outlier detection
3. Unit standardization to SI conventions
4. Non-plastic soil handling

---

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configuration
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Environment initialized successfully")

---
## 1. Data Ingestion
Load the unified raw dataset from Phase 1.

In [ ]:
# Load merged dataset
df = pd.read_csv('/home/claude/geotechnical_master_raw.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSource Distribution:")
print(df['Source_ID'].value_counts().sort_index())

# Display initial data quality
print(f"\n{'='*60}")
print("INITIAL DATA QUALITY ASSESSMENT")
print(f"{'='*60}")
print(df.info())

In [ ]:
# Visualize missing data patterns
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

if len(missing_pct) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    missing_pct.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Missing Data (%)')
    ax.set_title('Missing Data Analysis - Raw Dataset', fontsize=14, fontweight='bold')
    ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No missing data detected in the raw dataset.")

---
## 2. Atterberg Limits Consistency Logic

### Physical Relationship: $PI = LL - PL$

**Rules:**
- If exactly one parameter is missing, calculate it
- If two or more are missing, flag for imputation
- Handle "Non-Plastic" (NP) entries specially

In [ ]:
def handle_non_plastic(value):
    """Convert 'NP', 'Non-Plastic', etc. to 0."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        if value.upper().strip() in ['NP', 'NON-PLASTIC', 'NONPLASTIC', 'NON PLASTIC']:
            return 0.0
    return value

# Apply non-plastic handling to Atterberg columns
atterberg_cols = ['LL', 'PL', 'PI']

for col in atterberg_cols:
    if col in df.columns:
        df[col] = df[col].apply(handle_non_plastic)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Create is_plastic flag
if all(col in df.columns for col in atterberg_cols):
    df['is_plastic'] = ((df['LL'] != 0) | (df['PL'] != 0) | (df['PI'] != 0)).astype(int)
    print(f"Plastic soils: {df['is_plastic'].sum()}")
    print(f"Non-plastic soils: {(df['is_plastic'] == 0).sum()}")

In [ ]:
def enforce_atterberg_consistency(row):
    """
    Apply Atterberg consistency: PI = LL - PL
    
    Rules:
    - If exactly 1 missing: calculate it
    - If 2+ missing: flag for imputation
    - If all present but inconsistent: recalculate PI from LL and PL
    """
    LL, PL, PI = row['LL'], row['PL'], row['PI']
    
    missing_count = sum([pd.isna(x) for x in [LL, PL, PI]])
    
    # Case 1: All present - verify consistency
    if missing_count == 0:
        calculated_PI = LL - PL
        # Allow small tolerance for rounding errors
        if abs(PI - calculated_PI) > 1.0:
            row['PI'] = calculated_PI  # Trust LL and PL, recalculate PI
            row['atterberg_corrected'] = True
    
    # Case 2: Exactly one missing - calculate it
    elif missing_count == 1:
        if pd.isna(PI):
            row['PI'] = LL - PL
        elif pd.isna(LL):
            row['LL'] = PI + PL
        elif pd.isna(PL):
            row['PL'] = LL - PI
        row['atterberg_calculated'] = True
    
    # Case 3: Two or more missing - flag for imputation
    else:
        row['atterberg_needs_imputation'] = True
    
    return row

# Initialize tracking columns
df['atterberg_corrected'] = False
df['atterberg_calculated'] = False
df['atterberg_needs_imputation'] = False

# Apply consistency logic
if all(col in df.columns for col in atterberg_cols):
    df = df.apply(enforce_atterberg_consistency, axis=1)
    
    print(f"\n{'='*60}")
    print("ATTERBERG LIMITS CONSISTENCY REPORT")
    print(f"{'='*60}")
    print(f"Corrected (PI recalculated): {df['atterberg_corrected'].sum()}")
    print(f"Calculated (missing value derived): {df['atterberg_calculated'].sum()}")
    print(f"Needs imputation (2+ missing): {df['atterberg_needs_imputation'].sum()}")
    print(f"{'='*60}\n")

---
## 3. Physical Boundary Validation

Apply geotechnical constraints to filter unrealistic values.

In [ ]:
# Define physical boundaries for geotechnical parameters
PHYSICAL_LIMITS = {
    'Angle_Internal_Friction': (0, 50),      # degrees
    'Undrained_Cohesion': (0, 500),          # kPa (upper bound is conservative)
    'LL': (0, 150),                          # %
    'PL': (0, 100),                          # %
    'PI': (0, 100),                          # %
    'Moisture_Content': (0, 200),            # %
    'Fines_Content': (0, 100),               # %
    'Unit_Weight': (10, 25),                 # kN/m³
    'SPT_N': (0, 100),                       # blows/30cm
    'CPT_qc': (0, 50000),                    # kPa
}

def validate_physical_boundaries(df, limits_dict):
    """
    Flag and optionally remove records outside physical limits.
    """
    outlier_mask = pd.Series([False] * len(df), index=df.index)
    outlier_report = {}
    
    for col, (lower, upper) in limits_dict.items():
        if col in df.columns:
            # Identify outliers
            col_outliers = (df[col] < lower) | (df[col] > upper)
            col_outliers = col_outliers & df[col].notna()  # Only consider non-null
            
            if col_outliers.sum() > 0:
                outlier_report[col] = col_outliers.sum()
                outlier_mask |= col_outliers
    
    return outlier_mask, outlier_report

# Identify outliers
outlier_mask, outlier_report = validate_physical_boundaries(df, PHYSICAL_LIMITS)

print(f"\n{'='*60}")
print("PHYSICAL BOUNDARY VALIDATION")
print(f"{'='*60}")
if outlier_report:
    for param, count in outlier_report.items():
        print(f"{param:30s}: {count:4d} outliers detected")
    print(f"\nTotal records with outliers: {outlier_mask.sum()}")
else:
    print("No outliers detected - all values within physical bounds")
print(f"{'='*60}\n")

# Create clean dataset by removing outliers
df_clean = df[~outlier_mask].copy()
print(f"Records removed: {len(df) - len(df_clean)}")
print(f"Records retained: {len(df_clean)} ({100*len(df_clean)/len(df):.1f}%)")

---
## 4. Unit Standardization

Ensure all parameters follow SI conventions:
- Strength: kPa
- Unit Weight: kN/m³
- Angles: degrees

In [ ]:
# Unit conversion functions
def standardize_units(df):
    """
    Apply unit conversions to ensure SI consistency.
    
    Assumptions:
    - If Unit_Weight < 5, assume it's in g/cm³ → convert to kN/m³
    - If Undrained_Cohesion > 1000, assume it's in psf → convert to kPa
    """
    df = df.copy()
    
    # Unit Weight: g/cm³ to kN/m³ (multiply by 9.81)
    if 'Unit_Weight' in df.columns:
        low_unit_weight = df['Unit_Weight'] < 5
        if low_unit_weight.sum() > 0:
            df.loc[low_unit_weight, 'Unit_Weight'] *= 9.81
            print(f"Converted {low_unit_weight.sum()} Unit_Weight values from g/cm³ to kN/m³")
    
    # Cohesion: psf to kPa (divide by 20.885)
    if 'Undrained_Cohesion' in df.columns:
        high_cohesion = df['Undrained_Cohesion'] > 1000
        if high_cohesion.sum() > 0:
            df.loc[high_cohesion, 'Undrained_Cohesion'] /= 20.885
            print(f"Converted {high_cohesion.sum()} Cohesion values from psf to kPa")
    
    # Depth: feet to meters (if depth > 50, likely in feet)
    if 'Depth' in df.columns:
        depth_in_feet = df['Depth'] > 50
        if depth_in_feet.sum() > 0:
            df.loc[depth_in_feet, 'Depth'] *= 0.3048
            print(f"Converted {depth_in_feet.sum()} Depth values from feet to meters")
    
    return df

# Apply standardization
print(f"\n{'='*60}")
print("UNIT STANDARDIZATION")
print(f"{'='*60}")
df_clean = standardize_units(df_clean)
print(f"{'='*60}\n")

---
## 5. Data Quality Summary & Export

In [ ]:
# Generate comprehensive summary
print(f"\n{'='*60}")
print("CLEANED DATASET SUMMARY")
print(f"{'='*60}")
print(f"Original records: {len(df)}")
print(f"Cleaned records:  {len(df_clean)}")
print(f"Removal rate:     {100*(1-len(df_clean)/len(df)):.2f}%")
print(f"\nTarget Variable Availability:")
if 'Angle_Internal_Friction' in df_clean.columns:
    print(f"  Angle_Internal_Friction: {df_clean['Angle_Internal_Friction'].notna().sum()} samples")
if 'Undrained_Cohesion' in df_clean.columns:
    print(f"  Undrained_Cohesion:      {df_clean['Undrained_Cohesion'].notna().sum()} samples")
print(f"{'='*60}\n")

# Statistical summary of key parameters
key_params = ['Angle_Internal_Friction', 'Undrained_Cohesion', 'LL', 'PL', 'PI', 
              'Moisture_Content', 'Unit_Weight']
available_params = [p for p in key_params if p in df_clean.columns]

if available_params:
    print("\nDescriptive Statistics:")
    print(df_clean[available_params].describe().round(2))

In [ ]:
# Visualize target distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'Angle_Internal_Friction' in df_clean.columns:
    df_clean['Angle_Internal_Friction'].dropna().hist(bins=30, ax=axes[0], 
                                                        color='steelblue', edgecolor='black')
    axes[0].set_xlabel('Angle of Internal Friction (°)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of φ', fontweight='bold')
    axes[0].axvline(df_clean['Angle_Internal_Friction'].median(), 
                    color='red', linestyle='--', label='Median')
    axes[0].legend()

if 'Undrained_Cohesion' in df_clean.columns:
    df_clean['Undrained_Cohesion'].dropna().hist(bins=30, ax=axes[1], 
                                                  color='coral', edgecolor='black')
    axes[1].set_xlabel('Undrained Cohesion (kPa)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Cu', fontweight='bold')
    axes[1].axvline(df_clean['Undrained_Cohesion'].median(), 
                    color='red', linestyle='--', label='Median')
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Export cleaned dataset
output_path = '/home/claude/geotechnical_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"✓ CLEANED DATASET EXPORTED")
print(f"{'='*60}")
print(f"Location: {output_path}")
print(f"Shape:    {df_clean.shape}")
print(f"\nReady for Feature Engineering (Phase 3)")
print(f"{'='*60}\n")

---
## Summary

**Completed Operations:**
1. ✓ Atterberg limits consistency enforcement
2. ✓ Non-plastic soil handling with binary flag
3. ✓ Physical boundary outlier removal
4. ✓ Unit standardization to SI conventions
5. ✓ Clean dataset export

**Next Phase:** Feature_Engineering.ipynb